# 05 · Visualizaciones, reporte e implicaciones

**Objetivo:** Medir cuánto de una diferencia de precio entre municipios se mantiene cuando se comparan productos, presentaciones y periodos equivalentes, y distinguir lo que los datos públicos permiten afirmar de lo que haría falta para evaluar una oportunidad de negocio.

**Pregunta:** ¿Cómo se comunica el resultado y qué debe hacer cada audiencia con él?

Genera la gráfica principal (embudo) en tamaño grande y en las versiones para publicar, las gráficas de apoyo,
el README y el reporte HTML con selector de producto. Todo sale de `data/processed/` y de `resultados.json`.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
params = json.loads((META / "parametros.json").read_text(encoding="utf-8"))
REPS = params.get("bootstrap_replicas", 500)
import graficas as G, reporte as RP, plantillas
v = entorno.cifras()
rd = lambda f, **k: pd.read_csv(PROC / f, encoding="utf-8-sig", **k)
res = rd("embudo.csv")
prod = rd("brecha_producto_paso.csv").set_index("producto")
sob, casos_s = rd("sobrevivientes.csv"), rd("casos_sobreviven.csv")
casos, falta, sens = rd("casos_artefacto.csv"), rd("que_falta.csv"), rd("sensibilidad_embudo.csv")
display(res[["nombre", "brecha_mediana", "ic90_inf", "ic90_sup", "cambio_pp"]].round(1))

,nombre,brecha_mediana,ic90_inf,ic90_sup,cambio_pp
0,Comparación ingenua,25.2,23.4,28.3,NaN
1,Misma presentación,20.6,18.2,21.1,-4.6
2,Misma marca,17.7,16.6,18.9,-2.8
3,Mismo giro,17.2,15.8,18.2,-0.6
4,3 tiendas o más,12.1,10.9,13.3,-5.1
5,Mismo periodo,12.2,11.0,13.3,0.1
6,Sin errores de captura,12.2,11.0,13.3,0.0


## 1. Gráfica principal: el embudo

**Decisión:** barras horizontales en el orden del embudo, con la referencia de azar como línea dorada y el cambio de
cada paso en rojo ▼ (baja) o verde ▲ (sube). Alternativa descartada: un embudo «de embudo» con trapecios, que distorsiona
las magnitudes.

In [2]:
G.embudo(res, v, CHARTS / "01_embudo_comparabilidad.png", svg=True)
for fmt in C.SOCIAL:
    G.embudo_social(res, v, fmt, SOCIAL / f"01_embudo_comparabilidad_{fmt}.png")
plt.close("all")

## 2. Gráficas de apoyo

In [3]:
G.distribucion(prod, v, CHARTS / "02_distribucion_ingenua_validada.png")
G.artefactos(casos, CHARTS / "03_casos_artefacto.png")
G.sobrevivientes(sob, casos_s, v, CHARTS / "04_casos_sobreviven.png")
G.que_falta(falta, v, CHARTS / "05_que_falta.png")
plt.close("all")
print(sorted(p.name for p in CHARTS.glob("*.png")), sorted(p.name for p in SOCIAL.glob("*.png")))

['01_embudo_comparabilidad.png', '02_distribucion_ingenua_validada.png', '03_casos_artefacto.png', '04_casos_sobreviven.png', '05_que_falta.png'] ['01_embudo_comparabilidad_horizontal.png', '01_embudo_comparabilidad_vertical.png']


## 3. README y reporte HTML

In [4]:
gallery = [(plantillas.render(gr["titulo"], v), plantillas.render(gr["pie"], v), CHARTS / gr["archivo"]) for gr in CFG["graficas"]]
estructura = (ROOT / "src/estructura.md").read_text(encoding="utf-8")
(ROOT / "README.md").write_text(plantillas.readme(CFG, v, estructura), encoding="utf-8")
RP.build(CFG, v, res, prod, sob, casos, falta, sens, gallery, ROOT / "docs/index.html")
print("README y docs/index.html escritos")

README y docs/index.html escritos


## 4. Qué significa para el negocio

In [5]:
for aud, acc in CFG["readme"]["acciones"].items():
    display(Markdown(f"**{aud}**\n\n" + "\n".join(f"- {plantillas.render(a, v)}" for a in acc)))
display(Markdown(f'''**Conclusión.** La diferencia ingenua entre municipios ({v["paso_ingenua"]:.0f}%) baja a {v["validada"]:.1f}% al
comparar lo mismo, casi lo que daría el azar ({v["azar"]:.1f}%). Solo {v["n_sobreviven"]:.0f} de {v["n_articulos"]:,.0f} artículos
tienen una diferencia de municipio que supera la prueba, y aun en ellos faltan costos, márgenes y volumen para hablar
de oportunidad.'''))

**Director o dueño de cadena**

- **No abra una plaza ni mueva precios por una diferencia de un reporte.** A lo bruto, 44% de los productos parece costar más de 30% más en otro municipio; al comparar lo mismo, la diferencia típica es 12.2%, casi lo que daría el azar.
- **Pida las cinco preguntas antes de decidir:** ¿es el mismo artículo?, ¿cuántas tiendas hay detrás?, ¿supera al azar?, ¿la misma cadena cobra distinto?, ¿persiste en otra quincena?
- **Si una diferencia sobrevive, es para investigar con datos propios:** costo puesto en tienda, margen y volumen de esa plaza. Sin eso no hay caso de negocio.

**Equipo comercial o de ventas**

- **Mire las bebidas y los frescos.** Es donde sobreviven las diferencias (9 bebidas y 5 frescos relevantes) y donde la misma cadena cobra distinto por plaza: ahí hay precio por zona que conviene entender con el embotellador o el proveedor regional.
- **Compare el mismo artículo:** la presentación y la marca explican buena parte de las diferencias falsas. Una lista de precios por nombre genérico engaña.

**Analista**

- **Valide en este orden:** presentación, marca, giro, tiendas por municipio, fecha y errores. Exija 3 tiendas o más por municipio: con una sola, el municipio es esa tienda.
- **Compare contra el azar y corrija por pruebas múltiples.** Con más de mil artículos, unos 54 parecerían distintos solo por azar.
- **Siguiente paso, sin mezclarlo con PROFECO:** número de establecimientos por municipio (INEGI, DENUE), población (Censo de Población y Vivienda) y gasto de los hogares (ENIGH), para dimensionar las plazas donde la diferencia sobrevive.

**Conclusión.** La diferencia ingenua entre municipios (25%) baja a 12.2% al
comparar lo mismo, casi lo que daría el azar (11.5%). Solo 41 de 1,072 artículos
tienen una diferencia de municipio que supera la prueba, y aun en ellos faltan costos, márgenes y volumen para hablar
de oportunidad.